In [ ]:
!uv add requests langchain-huggingface chromadb langchain-text-splitters langchain-chroma

import os
import requests
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

# --- CONFIGURATION ---
# Replace with your actual API endpoint
API_URL = "https://api.yourcompany.com/v1/support-articles"
API_KEY = "API_KEY"
CHROMA_PATH = "./support_kb"  # Local folder to store the database

# 1. Initialize Hugging Face Embeddings
# This model converts text into numbers (vectors).
# 'all-mpnet-base-v2' is high quality. Use 'all-MiniLM-L6-v2' for faster performance.
print("Initializing Embedding Model...")
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    # model_kwargs={"token": API_KEY}
)


def fetch_api_data():
    """Fetch articles from your external support API."""
    print(f"Fetching data from {API_URL}...")

    # Example headers - adjust based on your API needs
    # response = requests.get(API_URL, headers={"Authorization": f"Bearer {API_KEY}"})
    # data = response.json()

    # MOCK DATA (Replace this with the real data from your response)
    data = [
        {
            "id": "101",
            "title": "How to Reset Password",
            "content": "To reset your password, click 'Forgot Password' on the login page and check your email for a reset link. The link expires in 24 hours.",
            "link": "https://help.yoursite.com/reset-password",
        },
        {
            "id": "102",
            "title": "Refund Policy",
            "content": "Our refund policy allows for full refunds within 30 days of purchase. Contact billing@yoursite.com with your order ID.",
            "link": "https://help.yoursite.com/refunds",
        },
    ]

    kb_articles = [
        {
            "id": "1",
            "title": "How to get started with Google Maps",
            "text": "To get started with Google Maps...",
            "link": "https://support.google.com/maps/answer/144349?hl=en&ref_topic=3092425&sjid=3974369040590557849-NC",
        },
        {
            "id": "2",
            "title": "How to use download areas and navigation offline in Google Maps",
            "text": "To download areas and use navigation offline...",
            "link": "https://support.google.com/maps/answer/6291838?hl=en&ref_topic=3092425&sjid=3974369040590557849-NC",
        },
        {
            "id": "3",
            "title": "How to Find & improve your location’s accuracy in Google Maps",
            "text": "To find and improve your location’s accuracy in Google Maps...",
            "link": "https://support.google.com/maps/answer/2839911?hl=en&ref_topic=3092425&sjid=3974369040590557849-NC",
        },
        {
            "id": "4",
            "title": "How to Add, edit, or delete Google Maps reviews & ratings",
            "text": "To use add, edit, or delete Google Maps reviews & ratings...",
            "link": "https://support.google.com/maps/answer/6230175?hl=en&ref_topic=3092425&sjid=3974369040590557849-NC",
        },
    ]
    return kb_articles


def process_and_load():
    # A. Get the raw data
    raw_articles = fetch_api_data()

    # B. Convert to LangChain Document objects
    documents = []
    for art in raw_articles:
        doc = Document(
            page_content=f"Title: {art['title']}\nContent: {art['text']}",
            metadata={
                "source_id": art["id"],
                "url": art["link"],
                "title": art["title"],
            },
        )
        documents.append(doc)

    # C. Split text into chunks
    # Why? LLMs have limits. Smaller chunks (500-1000 chars) are better for RAG.
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} articles into {len(chunks)} chunks.")

    # D. Save to local Vector DB (ChromaDB)
    # If the folder exists, it will add to it. If not, it creates it.
    print("Generating embeddings and saving to ChromaDB...")
    vectorstore = Chroma.from_documents(
        documents=chunks, embedding=embedding_model, persist_directory=CHROMA_PATH
    )

    print(f"Success! Database saved to {CHROMA_PATH}")


def test_query(query):
    """Small helper to test if the DB actually works."""
    db = Chroma(persist_directory=CHROMA_PATH, embedding_function=embedding_model)
    results = db.similarity_search(query, k=1)
    if results:
        print(f"\n--- TEST QUERY RESULT ---")
        print(f"Query: {query}")
        print(f"Match: {results[0].page_content}")
        print(f"Link: {results[0].metadata['url']}")


if __name__ == "__main__":
    process_and_load()

    # Test a search
    test_query("How can I find offline maps in Google Maps")

Resolved 197 packages in 14ms
Checked 194 packages in 58ms
Initializing Embedding Model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Fetching data from https://api.yourcompany.com/v1/support-articles...
Split 4 articles into 4 chunks.
Generating embeddings and saving to ChromaDB...
Success! Database saved to ./support_kb

--- TEST QUERY RESULT ---
Query: How can I find offline maps in Google Maps
Match: Title: How to use download areas and navigation offline in Google Maps
Content: To download areas and use navigation offline...
Link: https://support.google.com/maps/answer/6291838?hl=en&ref_topic=3092425&sjid=3974369040590557849-NC


In [142]:
import os
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint, HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# --- 1. SETUP LLM ---
os.environ["HUGGINGFACEHUB_API_TOKEN"] = API_KEY

llm_engine = HuggingFaceEndpoint(
    model="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation",
    max_new_tokens=512,
    temperature=0.01,
    cache=True
)
model = ChatHuggingFace(llm=llm_engine)

# --- 2. SETUP VECTOR DB AS RETRIEVER ---
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
vectorstore = Chroma(persist_directory="./support_kb", embedding_function=embeddings)

# Create a retriever that pulls the top 5 articles
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# --- 3. FORMATTING FUNCTION ---
# This function takes the DB results and formats them so the LLM clearly sees the URL
def format_docs(docs):
    formatted = []
    for i, d in enumerate(docs, 1):
        url = d.metadata.get('url', 'No Link Available')
        formatted.append(f"{i}. ARTICLE: {d.page_content}\n   LINK: {url}")
    return "\n\n--\n\n".join(formatted)

# --- 4. PROMPT TEMPLATE ---
prompt = ChatPromptTemplate.from_template("""
You are a helpful Customer Support Assistant.
Answer the user's question using ONLY the context provided below.
If the answer is not in the context, say "I am sorry, I do not have a guide for that."
You MUST include the exact LINK from the context at the end of your answer.

CONTEXT:
{context}

USER QUESTION:
{question}
                                          
Once you receive the tool results, return them in this exact format:

1. Article: <title>
   Link: <url>

2. Article: <title>
   Link: <url>

Do not add any extra commentary
""")

# --- 5. BUILD THE CHAIN (LangChain Expression Language) ---
# This reads from right to left / top to bottom:
# 1. Pass the question in.
# 2. Retrieve docs and format them.
# 3. Pass both into the Prompt.
# 4. Pass the Prompt to the LLM.
# 5. Output as a clean string.
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

# --- 6. RUN ---
def run_ticket(user_query: str):
    print(f"\nTicket: {user_query}")
    print("Searching DB and generating answer...")
    
    # Just call invoke on the chain!
    response = rag_chain.invoke(user_query)
    
    print("\n--- FINAL SUPPORT RESPONSE ---")
    print(response)

if __name__ == "__main__":
    run_ticket("How do I update my location on google map?")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Ticket: How do I update my location on google map?
Searching DB and generating answer...

--- FINAL SUPPORT RESPONSE ---
1. Article: How to Find & improve your location’s accuracy in Google Maps
   Link: https://support.google.com/maps/answer/2839911?hl=en&ref_topic=3092425&sjid=3974369040590557849-NC


In [154]:
import os
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint, HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# --- 1. SETUP LLM ---
os.environ["HUGGINGFACEHUB_API_TOKEN"] = API_KEY

llm_engine = HuggingFaceEndpoint(
    model="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation",
    max_new_tokens=512,
    temperature=0.01
)
model = ChatHuggingFace(llm=llm_engine)

# --- 2. SETUP VECTOR DB AS RETRIEVER ---
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
vectorstore = Chroma(persist_directory="./support_kb", embedding_function=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# --- 3. FORMATTING FUNCTION ---
def format_docs(docs):
    """Format retrieved documents for the LLM, removing duplicates."""
    if not docs:
        return "No articles found."
    
    # Remove duplicates based on URL
    seen_urls = set()
    formatted = []
    counter = 1
    
    for doc in docs:
        url = doc.metadata.get("url", "No link available")
        if url in seen_urls:
            continue
        seen_urls.add(url)
        title = doc.metadata.get("title", "Untitled")
        formatted.append(f"{counter}. Article: {title}\n   Link: {url}")
        counter += 1
    
    return "\n\n".join(formatted)

# --- 4. PROMPT TEMPLATE ---
prompt = ChatPromptTemplate.from_template("""You are a Customer Support Agent.
Use ONLY the provided articles to answer the question.
If no articles are relevant, say "I'm sorry, I don't have a guide for that."

CONTEXT (Support Articles):
{context}

USER QUESTION:
{question}

Return the answer in this format:

1. Article: <title>
   Link: <url>

2. Article: <title>
   Link: <url>

Do not add any extra commentary.""")

# --- 5. BUILD RAG CHAIN ---
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

# --- 6. RUN ---
def run_ticket(user_query: str):
    print(f"\n🎫 Ticket: '{user_query}'\n")
    response = rag_chain.invoke(user_query)
    print("--- Support Articles Found ---\n")
    print(response)

run_ticket("How do I update my location on google map?")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🎫 Ticket: 'How do I update my location on google map?'

--- Support Articles Found ---

1. Article: How to Find & improve your location’s accuracy in Google Maps
   Link: https://support.google.com/maps/answer/2839911?hl=en&ref_topic=3092425&sjid=3974369040590557849-NC
